# Ejercicio — Word Embeddings con Gensim

**Corpus:** 10 frases del Decálogo del Politécnico

In [1]:
!pip install gensim

import re
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Porque aspiro a ser todo un hombre",
    "Porque exijo mis deberes antes que mis derechos",
    "Por convicción y no por circunstancia",
    "Para alcanzar las conquistas universales y ofrecerlas a mi pueblo",
    "Porque me duele la Patria en mis entrañas y aspiro a calmar sus dolencias",
    "Porque ardo en deseos de despertar al hermano dormido",
    "Para prender una antorcha en el altar de la Patria",
    "Porque me dignifico y siento el deber de dignificar a mi institución",
    "Porque mi respetada libertad de joven y estudiante me impone la razón de respetar este recinto",
    "Porque traduzco la tricromía de mi bandera como trabajo deber y honor"
]

print("Corpus:")
for i, doc in enumerate(documents):
    print(f"  D{i+1}: \"{doc}\"")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 17.3 MB/s eta 0:00:0000:0100:01
Corpus:
  D1: "Porque aspiro a ser todo un hombre"
  D2: "Porque exijo mis deberes antes que mis derechos"
  D3: "Por convicción y no por circunstancia"
  D4: "Para alcanzar las conquistas universales y ofrecerlas a mi pueblo"
  D5: "Porque me duele la Patria en mis entrañas y aspiro a calmar sus dolencias"
  D6: "Porque ardo en deseos de despertar al hermano dormido"
  D7: "Para prender una antorcha en el altar de la Patria"
  D8: "Porque me dignifico y siento el deber de dignificar a mi institución"
  D9: "Porque mi respetada libertad de joven y estudiante me impone la razón de respetar este recinto"
  D10: "Porque traduzco la tricromía de mi bandera como trabajo deber y honor"


In [2]:
# Preprocesamiento
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

tokenized_docs = [preprocess(doc) for doc in documents]

print("Documentos tokenizados:")
for i, tokens in enumerate(tokenized_docs):
    print(f"  D{i+1}: {tokens}")

Documentos tokenizados:
  D1: ['porque', 'aspiro', 'a', 'ser', 'todo', 'un', 'hombre']
  D2: ['porque', 'exijo', 'mis', 'deberes', 'antes', 'que', 'mis', 'derechos']
  D3: ['por', 'convicción', 'y', 'no', 'por', 'circunstancia']
  D4: ['para', 'alcanzar', 'las', 'conquistas', 'universales', 'y', 'ofrecerlas', 'a', 'mi', 'pueblo']
  D5: ['porque', 'me', 'duele', 'la', 'patria', 'en', 'mis', 'entrañas', 'y', 'aspiro', 'a', 'calmar', 'sus', 'dolencias']
  D6: ['porque', 'ardo', 'en', 'deseos', 'de', 'despertar', 'al', 'hermano', 'dormido']
  D7: ['para', 'prender', 'una', 'antorcha', 'en', 'el', 'altar', 'de', 'la', 'patria']
  D8: ['porque', 'me', 'dignifico', 'y', 'siento', 'el', 'deber', 'de', 'dignificar', 'a', 'mi', 'institución']
  D9: ['porque', 'mi', 'respetada', 'libertad', 'de', 'joven', 'y', 'estudiante', 'me', 'impone', 'la', 'razón', 'de', 'respetar', 'este', 'recinto']
  D10: ['porque', 'traduzco', 'la', 'tricromía', 'de', 'mi', 'bandera', 'como', 'trabajo', 'deber', 'y', 'h

In [3]:
# a) Crear word embeddings con Word2Vec
model = Word2Vec(
    sentences=tokenized_docs,
    vector_size=50,      # dimensión de los vectores (50 es adecuado para corpus pequeño)
    window=5,            # palabras de contexto a considerar
    min_count=1,         # incluir todas las palabras (min_count=1 porque el corpus es pequeño)
    workers=4,
    epochs=200,          # muchas épocas para compensar el corpus pequeño
    seed=42
)

print(f"Modelo Word2Vec entrenado")
print(f"Vocabulario: {len(model.wv)} palabras")
print(f"Dimensión de vectores: {model.wv.vector_size}")
print(f"\nEjemplo — vector de 'patria' (primeros 10 valores):")
print(f"  {model.wv['patria'][:10].round(4)}")

Modelo Word2Vec entrenado
Vocabulario: 67 palabras
Dimensión de vectores: 50

Ejemplo — vector de 'patria' (primeros 10 valores):
  [ 0.0462 -0.0543  0.0237 -0.0272  0.0456 -0.0203 -0.0136  0.0404 -0.0389
  0.0125]


In [5]:
# b) Calcular vectores de documento (promedio de los word embeddings)
#    y matriz de similitud coseno entre documentos

def document_vector(tokens, model):
    """Representa un documento como el promedio de sus word embeddings."""
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.wv.vector_size)
    return np.mean(vectors, axis=0)

# Obtener vectores de todos los documentos
doc_vectors = np.array([document_vector(tokens, model) for tokens in tokenized_docs])

# Calcular matriz de similitud coseno
sim_matrix = cosine_similarity(doc_vectors)

# Mostrar como tabla
labels = [f"D{i+1}" for i in range(len(documents))]
print("Matriz de similitud coseno entre documentos:\n")

# Header
print(f"{'':>5}", end="")
for label in labels:
    print(f"{label:>8}", end="")
print()

# Rows
for i, label in enumerate(labels):
    print(f"{label:>5}", end="")
    for j in range(len(labels)):
        print(f"{sim_matrix[i][j]:>8.4f}", end="")
    print()

Matriz de similitud coseno entre documentos:

           D1      D2      D3      D4      D5      D6      D7      D8      D9     D10
   D1  1.0000  0.9893  0.9836  0.9941  0.9967  0.9930  0.9951  0.9957  0.9959  0.9947
   D2  0.9893  1.0000  0.9840  0.9895  0.9937  0.9896  0.9898  0.9921  0.9921  0.9898
   D3  0.9836  0.9840  1.0000  0.9869  0.9879  0.9853  0.9857  0.9859  0.9893  0.9873
   D4  0.9941  0.9895  0.9869  1.0000  0.9960  0.9933  0.9967  0.9972  0.9967  0.9969
   D5  0.9967  0.9937  0.9879  0.9960  1.0000  0.9937  0.9971  0.9975  0.9980  0.9960
   D6  0.9930  0.9896  0.9853  0.9933  0.9937  1.0000  0.9936  0.9943  0.9957  0.9931
   D7  0.9951  0.9898  0.9857  0.9967  0.9971  0.9936  1.0000  0.9975  0.9973  0.9979
   D8  0.9957  0.9921  0.9859  0.9972  0.9975  0.9943  0.9975  1.0000  0.9979  0.9973
   D9  0.9959  0.9921  0.9893  0.9967  0.9980  0.9957  0.9973  0.9979  1.0000  0.9971
  D10  0.9947  0.9898  0.9873  0.9969  0.9960  0.9931  0.9979  0.9973  0.9971  1.0000


In [ ]:
# c) Documentos más similares y más disímiles

n = len(documents)
max_sim = -1
min_sim = 2
most_similar_pair = (0, 0)
most_dissimilar_pair = (0, 0)

for i in range(n):
    for j in range(i + 1, n):
        if sim_matrix[i][j] > max_sim:
            max_sim = sim_matrix[i][j]
            most_similar_pair = (i, j)
        if sim_matrix[i][j] < min_sim:
            min_sim = sim_matrix[i][j]
            most_dissimilar_pair = (i, j)

i, j = most_similar_pair
print("=" * 60)
print("DOCUMENTOS MÁS SIMILARES:")
print(f"  D{i+1}: \"{documents[i]}\"")
print(f"  D{j+1}: \"{documents[j]}\"")
print(f"  Similitud coseno: {max_sim:.4f}")

i, j = most_dissimilar_pair
print(f"\nDOCUMENTOS MÁS DISÍMILES:")
print(f"  D{i+1}: \"{documents[i]}\"")
print(f"  D{j+1}: \"{documents[j]}\"")
print(f"  Similitud coseno: {min_sim:.4f}")
print("=" * 60)

DOCUMENTOS MÁS SIMILARES:
  D5: "Porque me duele la Patria en mis entrañas y aspiro a calmar sus dolencias"
  D9: "Porque mi respetada libertad de joven y estudiante me impone la razón de respetar este recinto"
  Similitud coseno: 0.9980

DOCUMENTOS MÁS DISÍMILES:
  D1: "Porque aspiro a ser todo un hombre"
  D3: "Por convicción y no por circunstancia"
  Similitud coseno: 0.9836
